In [ ]:
#This code is to add still not officially announced results by State of 2024 (as of early January 2025) and to do the cleaning
#of election data from 1976 to 2020 (we are interested in data from 2020).
import pandas as pd
import numpy as np
#Loading Economic data from 2020 to 2023
df2020= pd.read_csv("df_new20.csv")
#Loading Election data between 1976-2020
tott = pd.read_csv("1976-2020-president.csv")


In [ ]:
#Election estimates 2024. The official results havent been published yet. Taken from state government sites
stateres24 = [(49.81,48.33),(64.6, 34.1),(54.5, 41.4),(52.2,46.7),(64.2,33.6),(38.3,58.5),(43.1,54.1),(41.9,56.4),
            (41.8, 56.5),(6.5,90.3),(56.1,43.0),(50.7,48.5),(37.5,60.6),(66.9,30.4),(43.5,54.4),(58.8,39.4),
            (55.7,42.5),(57.2,41.0),(64.5,33.9),(60.2,38.2),(45.5,52.4),(34.1,62.6),(36.0,61.2),(49.7,48.3),(46.7,50.9),
            (60.9,38.0),(58.5,40.1),(58.4,38.5),(59.3,38.9),(50.6,47.5),(47.9,50.7),(46.1,52.0),(45.9,51.9),(43.3,55.9),
            (50.9,47.7),(67.0,30.5),(55.1,43.9),(66.2,31.9),(41.0,55.3),(50.4,48.7),(41.8,55.5),(58.2,40.4),(63.4,34.2),(64.2,34.5),
             (56.1,42.5),(59.4,37.8),(32.0,63.2),(46.1,51.8),(39.0,57.2),(70.0,28.1),(49.6,48.7),(71.6,25.8)]
statelist = list(df2020["GeoName"])
res24df = pd.DataFrame(stateres24, columns=["REPUBLICAN", "DEMOCRAT"], index=(statelist))
res24df.head(2)

,REPUBLICAN,DEMOCRAT
UNITED STATES,49.81,48.33
ALABAMA,64.60,34.10


In [ ]:
    #Cleaning the presidential elections 1976-2020 data
#We search for all names not related to a party, in order to get the name of invalid votes
nanrow = tott[tott["party_detailed"].isna()]
nanrow["candidate"].unique() #We can see a list that contains some invalid names

notcandidate = ['SCATTERING','BLANK VOTE/SCATTERING','NONE OF THE ABOVE','NONE OF THESE CANDIDATES',
                'BLANK VOTE', 'OVER VOTE','BLANK VOTE/VOID VOTE/SCATTERING',
                'VOID VOTE','BLANK VOTES', 'OVERVOTES','UNDERVOTES', 'VOID']
#We get the index of notcandidate to sort of invalid votes
invalid = tott[tott["candidate"].isin(notcandidate)].index
tott.drop(invalid, inplace=True)

#We drop the columns we don't need. Also totalvotes because it takes in consideration invalid votes
tott = tott.drop(tott.columns[2:8], axis=1)
tott = tott.drop(["writein","version","notes","totalvotes","party_detailed"], axis=1)

display(tott)


,year,state,candidatevotes,party_simplified
0,1976,ALABAMA,659170,DEMOCRAT
1,1976,ALABAMA,504070,REPUBLICAN
2,1976,ALABAMA,9198,OTHER
3,1976,ALABAMA,6669,OTHER
4,1976,ALABAMA,1954,OTHER
...,...,...,...,...
4280,2020,WYOMING,73491,DEMOCRAT
4281,2020,WYOMING,193559,REPUBLICAN
4282,2020,WYOMING,5768,LIBERTARIAN
4283,2020,WYOMING,2208,OTHER


In [ ]:
#We create a more reliable total votes column
sumvotes = tott.groupby(["year","state"])["candidatevotes"].agg(sum)


<ipython-input-4-b28e4fc7cf35>:2: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  sumvotes = tott.groupby(["year","state"])["candidatevotes"].agg(sum)


In [ ]:
tott = tott.merge(sumvotes.rename("totalvotes"), on=["year","state"])


In [ ]:
#We consider only Democratic and Republican party
todrop = tott[tott["party_simplified"].isin(["OTHER","LIBERTARIAN"])].index
tott.drop(todrop, inplace=True)

#And we add a column for party percentage
tott["percentage"]=tott["candidatevotes"]/tott["totalvotes"]

display(tott.tail(7))

,year,state,candidatevotes,party_simplified,totalvotes,percentage
4185,2020,WASHINGTON,1584651,REPUBLICAN,4087631,0.387670
4191,2020,WEST VIRGINIA,235984,DEMOCRAT,794652,0.296965
4192,2020,WEST VIRGINIA,545382,REPUBLICAN,794652,0.686316
4195,2020,WISCONSIN,1630866,DEMOCRAT,3298041,0.494495
4196,2020,WISCONSIN,1610184,REPUBLICAN,3298041,0.488224
4208,2020,WYOMING,73491,DEMOCRAT,276765,0.265536
4209,2020,WYOMING,193559,REPUBLICAN,276765,0.699362


In [ ]:
#We create a row for the whole USA
ustott = tott.groupby(["year","party_simplified"]).agg(
    {"totalvotes": "sum", "candidatevotes":"sum"})
ustott.reset_index(inplace=True)
ustott["state"]="United States"
ustott["percentage"]=ustott["candidatevotes"]/ustott["totalvotes"]
#and we add it to our states dataframe
tott = pd.concat([tott, ustott], axis=0)

In [ ]:
#We can transform the dataframe to get only the party percentages
tott = tott.drop(["candidatevotes", "totalvotes"], axis=1)
transformed = tott.pivot_table(index=["year", "state"], columns=["party_simplified"])


In [ ]:
filetransformed = transformed.to_csv("election_percentages_76_20.csv")
res24df.to_csv("election_percentages_24.csv")